# Pycox: DeepSurv Stratified by Batch


In [1]:
import os
os.getcwd()

'/home/nfs/dengy/dl-survival-miRNA/scripts/examples'

In [2]:
import os
import numpy as np
import torch
import torchtuples as tt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn_pandas import DataFrameMapper

os.chdir("../..")
from pss.pycox.models import CoxPH, CoxPHStratified, StratifiedDataset
from pss.pycox.evaluation.eval_surv import EvalSurv
from pss.utils import load_prepared_split #load_simulate_survival_data
from pss.run_models import DeepSurvPipeline, train_over_subsets

## *Test: Debugging*

### *Network code*

In [3]:
import numpy as np
import torch
from torch import Tensor

def cox_ph_loss_sorted(log_h: Tensor, events: Tensor, eps: float = 1e-7) -> Tensor:
    """Requires the input to be sorted by descending duration time.
    See DatasetDurationSorted.

    We calculate the negative log of $(\frac{h_i}{\sum_{j \in R_i} h_j})^d$,
    where h = exp(log_h) are the hazards and R is the risk set, and d is event.

    We just compute a cumulative sum, and not the true Risk sets. This is a
    limitati`on, but simple and fast.
    """
    if events.dtype is torch.bool:
        events = events.float()
    events = events.view(-1)
    log_h = log_h.view(-1)
    if events.sum() == 0:
        return log_h.sum() * 0.0  # update 08/11/25: safe dummy loss
    
    gamma = log_h.max()
    log_cumsum_h = log_h.sub(gamma).exp().cumsum(0).add(eps).log().add(gamma)
    return - log_h.sub(log_cumsum_h).mul(events).sum().div(events.sum())


def cox_ph_loss(log_h: Tensor, durations: Tensor, events: Tensor, eps: float = 1e-7) -> Tensor:
    """Loss for CoxPH model. If data is sorted by descending duration, see `cox_ph_loss_sorted`.

    We calculate the negative log of $(\frac{h_i}{\sum_{j \in R_i} h_j})^d$,
    where h = exp(log_h) are the hazards and R is the risk set, and d is event.

    We just compute a cumulative sum, and not the true Risk sets. This is a
    limitation, but simple and fast.
    """
    idx = durations.sort(descending=True)[1]
    events = events[idx]
    log_h = log_h[idx]
    return cox_ph_loss_sorted(log_h, events, eps)


####### [UPDATE] 07/07/2025
def stratified_cox_ph_loss(log_h: Tensor, durations: Tensor, events: Tensor, batch_indices: Tensor, eps: float = 1e-7) -> Tensor:
    """
    Stratified CoxPH loss that computes partial likelihood across batches.

    Arguments:
        log_h {torch.Tensor} -- Log hazard predictions for each instance.
        durations {torch.Tensor} -- Duration times for each instance.
        events {torch.Tensor} -- Event indicators (1 if event, 0 if censored).
        batch_indices {numpy array} -- Batch labels for each instance.
        eps {float} -- Small epsilon for numerical stability.

    Returns:
        torch.Tensor -- The total stratified negative log partial likelihood.
    """
    device = batch_indices.device
    unique_batches = torch.unique(batch_indices)
    losses = torch.zeros(len(unique_batches), device=device)
    n_valid_batch = 0
        
    for i, batch in enumerate(unique_batches):
        # Select data for the current batch
        mask = (batch_indices == batch)
        if mask.sum() == 0 or events[mask].sum() == 0:
            continue  # skip empty batch (added 08/11/25) or batch with no events
        
        # Sort by descending durations
        idx = torch.argsort(durations[mask], descending=True)
        
        events_batch = events[mask][idx]
        log_h_batch = log_h[mask][idx]
        if events_batch.sum() == 0:
            continue 
        
        losses[i] = cox_ph_loss_sorted(log_h_batch, events_batch, eps)
        n_valid_batch += 1
        
    if n_valid_batch == 0:
        return log_h.sum() * 0.0
    # print(n_valid_batch)
    return losses.sum()

In [4]:
# Create a PyTorch tensor
batch_indices = torch.tensor([1, 2, 2, 2, 2, 2, 3, 3, 4, 4], dtype=torch.float32)
durations = torch.tensor([169.5, 0.6, 12.3, 1.5, 3.8, 0.1, 0.1, 0.1, 0.6, 0.1], dtype=torch.float32)
events = torch.tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.float32)
log_h = torch.tensor([-4.1238, 2.1188, -1.5863, -1.2239, 0.9088, 5.6637, 1.2920, 4.5356, 1.5392, 5.0004], dtype=torch.float32)

# Test out ufnction
device = batch_indices.device
unique_batches = torch.unique(batch_indices)
losses = torch.zeros(len(unique_batches), device=device)

for i, batch in enumerate(unique_batches):
    # i = 1
    # batch = 1
    # print(i)
    mask = (batch_indices == batch)
    if mask.sum() == 0:
        print(f"batch {batch} is empty")
        continue
    idx = torch.argsort(durations[mask], descending=True)
    # idx = durations[mask].sort(descending=True)[1]
    log_h_batch = log_h[mask][idx]
    events_batch = events[mask][idx]
    
    print(events_batch)
    if events_batch.sum() == 0:
        print(f"batch {int(batch)} has no events")
        continue
    
    losses[i] = cox_ph_loss_sorted(log_h_batch, events_batch, eps=1e-7)
    
losses.sum()

tensor([0.])
batch 1 has no events
tensor([1., 1., 1., 1., 1.])
tensor([1., 1.])
tensor([1., 1.])


tensor(0.5826)

In [5]:
stratified_cox_ph_loss(log_h, durations, events, batch_indices)

tensor(0.5826)

# *Test: Full steps*

### Load Train/Test Data

In [6]:
batchNormType='BE11Asso00_normNone'
dataType='linear-p10'
train_size=1000
random_state=42
time_col='time'
status_col='status'
batch_col='batch_id'
iter_i = 1

train_df, test_df = load_prepared_split(batchNormType=batchNormType,
                                        dataName=dataType,
                                        keep_batch=True,
                                        train_size=train_size,
                                        iter_i=iter_i)

print(f"Training data dimensions: {train_df.shape}")
print(f"Testing data dimensions:  {test_df.shape}")

Training data dimensions: (1000, 541)
Testing data dimensions:  (1000, 541)


### Create DL object & hyperparameter space

In [7]:
hyperparameters = {
    "num_nodes": {"type": "categorical", "choices": [
        "128",
        "64",
        "32",
        "32-16",
        "64-32",
        "128-64",
        "64-64-32",
        "32-32-16"
        ]},
    "dropout": {"type": "float", "low": 0.1, "high": 0.5},
    "weight_decay": {"type": "float", "low": 1e-5, "high": 1e-2, "log": True},
    "learning_rate": {"type": "float", "low": 1e-4, "high": 1e-2, "log": True},
    "batch_size": {"type": "categorical", "choices": [128, 64, 32, 16]}
}
dl = DeepSurvPipeline(
    train_df=None, test_df=None,
    batchNormType=batchNormType,
    dataName=dataType,
    time_col=time_col,
    status_col=status_col,
    batch_col=batch_col,
    hyperparameters=hyperparameters,
    is_stratified=True,
    storage_url="sqlite:///deepsurv-torch-hp-test.db"
)

In [11]:
# # optuna database check
# import optuna
# optuna.get_all_study_names(storage="sqlite:///deepsurv-torch-hp-test.db")
# optuna.delete_study(storage="sqlite:///deepsurv-torch-hp-test.db", study_name='BE11Asso00_normNone-linear-p10-stratified-deepsurv-torch-1000')

In [12]:
# Tune via optuna for automatic trials
n_splits=5
n_trials=30
trial_threshold=30
n_jobs=1

dl.tune_hyperparameters(
    train_df,
    n_samples=train_size,
    n_splits=n_splits,
    n_trials=n_trials, 
    trial_threshold=trial_threshold,
    n_jobs=n_jobs
)

[I 2026-05-13 12:57:54,127] A new study created in RDB with name: BE11Asso00_normNone-linear-p10-stratified-deepsurv-torch-1000


⚠️No completed trials in Optuna study 'BE11Asso00_normNone-linear-p10-stratified-deepsurv-torch-1000'. Start hyperparameter tuning...


[I 2026-05-13 12:58:20,327] Trial 0 finished with value: 0.780188709388007 and parameters: {'num_nodes': '64', 'dropout': 0.3651208970997897, 'weight_decay': 0.00028487539871876447, 'learning_rate': 0.00012922454924988965, 'batch_size': 64}. Best is trial 0 with value: 0.780188709388007.
[I 2026-05-13 12:58:37,715] Trial 1 finished with value: 0.7877113193618969 and parameters: {'num_nodes': '128-64', 'dropout': 0.22934382991058275, 'weight_decay': 0.007837687063132745, 'learning_rate': 0.0009073813629975028, 'batch_size': 64}. Best is trial 1 with value: 0.7877113193618969.
[I 2026-05-13 12:59:40,423] Trial 2 finished with value: 0.7724172738072793 and parameters: {'num_nodes': '128-64', 'dropout': 0.18149112095710002, 'weight_decay': 0.00011490381894797433, 'learning_rate': 0.0014379036455112678, 'batch_size': 16}. Best is trial 1 with value: 0.7877113193618969.
[I 2026-05-13 13:00:19,517] Trial 3 finished with value: 0.7814910557465635 and parameters: {'num_nodes': '32', 'dropout': 

In [13]:
def _preprocess_data(df, mapper=None, fit_scaler=True):
    survival_cols = [time_col, status_col]
    covariate_cols = [col for col in df.columns if col not in survival_cols]
    # Transform features (miRNA expression)
    if fit_scaler or mapper is None:
        standardize = [([col], StandardScaler()) for col in covariate_cols]
        mapper = DataFrameMapper(standardize)
        x = mapper.fit_transform(df[covariate_cols]).astype('float32')
    else:
        x = mapper.transform(df[covariate_cols]).astype('float32')
    # Prepare labels (survival data)
    y = (df[time_col].values, df[status_col].values)
    
    return x, y, mapper

batch_ids_train = train_df[batch_col].to_numpy().reshape(-1)
batch_ids_test = test_df[[batch_col]].to_numpy().reshape(-1)

train_sub = train_df.drop(columns=[batch_col])
test_sub = test_df.drop(columns=[batch_col])

x_train, y_train, mapper = _preprocess_data(train_sub)
x_test, y_test, _ = _preprocess_data(test_sub, mapper=mapper, fit_scaler=False)

durations_train, events_train = y_train[0], y_train[1]
durations_test, events_test = y_test[0], y_test[1]

# Prepare data 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x_train = torch.from_numpy(x_train).to(device)
x_test = torch.from_numpy(x_test).to(device)

batch_ids_train = torch.from_numpy(batch_ids_train).long().to(device)
batch_ids_test   = torch.from_numpy(batch_ids_test).long().to(device)
durations_train = torch.from_numpy(durations_train).float().to(device)
durations_test = torch.from_numpy(durations_test).float().to(device)
events_train = torch.from_numpy(events_train).float().to(device)
events_test = torch.from_numpy(events_test).float().to(device)
y_train = (durations_train, events_train)
y_test = (durations_test, events_test)

print(device)
print(x_test.shape)           # Should be [n_samples, n_features]
print(durations_test.shape)   # Should be [n_samples]
print(events_test.shape)      # Should be [n_samples]
print(batch_ids_test.shape)   # Should be [n_samples]

cpu
torch.Size([1000, 538])
torch.Size([1000])
torch.Size([1000])
torch.Size([1000])


In [14]:
def _parse_num_nodes(s):
    return [int(x) for x in s.split("-")]

params = dl._best_params
input_size = x_train.shape[1]
output_size = 1
num_nodes = params.get("num_nodes", "32-16")            # Default num of layers & nodes
num_nodes = _parse_num_nodes(num_nodes)
dropout = params.get("dropout", 0.1)                    # Default dropout rate
learning_rate = params.get("learning_rate", 1e-3)       # Default learning rate
batch_size = params.get("batch_size", 128)              # Default batch size
epochs = params.get("epochs", 500)                      # Default number of epochs
batch_norm = params.get("batch_norm", True)             # Default batch normalization
output_bias = params.get("output_bias", True)           # Default output bias
weight_decay = params.get("weight_decay", 1e-4)         # Default weight decay
activation_map = {                                      
    "ReLU": torch.nn.ReLU,
    "LeakyReLU": torch.nn.LeakyReLU,
    "SELU": torch.nn.SELU
} 
activation = activation_map.get(params.get("activation", "ReLU")) # Activation function

In [23]:
net = tt.practical.MLPVanilla(
    in_features=input_size,
    out_features=output_size,
    num_nodes=num_nodes,
    dropout=dropout, 
    batch_norm=batch_norm,
    activation=activation,
    output_bias=output_bias
).to(device)
optimizer = tt.optim.Adam(weight_decay=weight_decay, lr=learning_rate)

# Get default early stopping settings if not defined 
patience = 30
min_delta = 1e-3
callbacks = [tt.callbacks.EarlyStopping(patience=patience, min_delta=min_delta)]

### CoxPH

In [24]:
# CoxPH model
model_nonstrat = CoxPH(net, optimizer=optimizer)
log = model_nonstrat.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    callbacks=callbacks, 
    verbose=True,
    val_data=(x_test, y_test),
    val_batch_size=batch_size
)

0:	[0s / 0s],		train_loss: 3.4610,	val_loss: 3.1939
1:	[0s / 0s],		train_loss: 3.2177,	val_loss: 3.1196
2:	[0s / 0s],		train_loss: 3.1064,	val_loss: 3.0234
3:	[0s / 0s],		train_loss: 3.0235,	val_loss: 2.9677
4:	[0s / 0s],		train_loss: 2.9966,	val_loss: 2.9559
5:	[0s / 0s],		train_loss: 3.0035,	val_loss: 2.9935
6:	[0s / 0s],		train_loss: 3.0137,	val_loss: 2.9338
7:	[0s / 0s],		train_loss: 3.0009,	val_loss: 2.9261
8:	[0s / 0s],		train_loss: 2.9643,	val_loss: 2.9199
9:	[0s / 0s],		train_loss: 2.9425,	val_loss: 2.9043
10:	[0s / 0s],		train_loss: 2.9263,	val_loss: 2.9082
11:	[0s / 0s],		train_loss: 2.8691,	val_loss: 2.8959
12:	[0s / 0s],		train_loss: 2.9132,	val_loss: 2.8965
13:	[0s / 0s],		train_loss: 2.8848,	val_loss: 2.8806
14:	[0s / 0s],		train_loss: 2.8744,	val_loss: 2.8817
15:	[0s / 0s],		train_loss: 2.8924,	val_loss: 2.8773
16:	[0s / 0s],		train_loss: 2.8172,	val_loss: 2.8686
17:	[0s / 0s],		train_loss: 2.8259,	val_loss: 2.9583
18:	[0s / 0s],		train_loss: 2.9213,	val_loss: 2.9527
19:

In [25]:
# ==================== Evaluation ====================
_ = model_nonstrat.compute_baseline_hazards(input=x_train, target=(durations_train, events_train))

# Convert torch tensors back to numpy objects for evaluation
x_train_np = x_train.detach().cpu().numpy()
x_test  = x_test.detach().cpu().numpy()
durations_train = durations_train.detach().cpu().numpy()
durations_test  = durations_test.detach().cpu().numpy()
events_train    = events_train.detach().cpu().numpy()
events_test     = events_test.detach().cpu().numpy()

# Initialize EvalSurv objects 
tr_surv  = model_nonstrat.predict_surv_df(x_train)
te_surv = model_nonstrat.predict_surv_df(x_test)
tr_ev = EvalSurv(tr_surv, durations_train, events_train, censor_surv='km')
te_ev = EvalSurv(te_surv, durations_test, events_test, censor_surv='km')

# Concordance index ----------------
tr_c_index  = tr_ev.concordance_td() 
te_c_index = te_ev.concordance_td() 

tr_c_index, te_c_index

((0.838706175030802, 423674.0), (0.7985209264154014, 436219.0))

In [ ]:
# Compute baseline hazards (per-batch)
_ = model_nonstrat.compute_baseline_hazards(input=x_train, target=(durations_train, events_train), batch_ids=None)

# Initialize EvalSurv objects 
tr_surv  = model_nonstrat.predict_surv_df(x_train)
te_surv = model_nonstrat.predict_surv_df(x_test)
tr_ev = EvalSurv(tr_surv, durations_train, events_train, censor_surv='km')
te_ev = EvalSurv(te_surv, durations_test, events_test, censor_surv='km')

# Concordance index ----------------
tr_strat_c_index  = tr_ev.stratified_concordance_td(batch_indices=batch_ids_train) 
te_strat_c_index = te_ev.stratified_concordance_td(batch_indices=batch_ids_test) 

print(tr_strat_c_index, te_strat_c_index)

0.8383892521527853 0.7982938718662953


### Stratified CoxPH

In [16]:
train_dataset = StratifiedDataset(x_train, durations_train, events_train, batch_ids_train)
test_dataset = StratifiedDataset(x_test, durations_test, events_test, batch_ids_test)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

for xb, db, eb, bb in test_loader:
    print("VAL batch total events:", int(eb.sum().item()),
          "| per-stratum:", {int(s): int(eb[bb==s].sum().item()) for s in bb.unique().tolist()})

VAL batch total events: 48 | per-stratum: {1: 5, 2: 2, 3: 5, 4: 1, 5: 2, 6: 6, 7: 6, 8: 3, 9: 1, 10: 3, 11: 5, 12: 4, 13: 4, 14: 1, 15: 0}
VAL batch total events: 48 | per-stratum: {1: 1, 2: 2, 3: 6, 4: 5, 5: 5, 6: 1, 7: 3, 8: 5, 9: 2, 10: 3, 11: 3, 12: 3, 13: 4, 14: 2, 15: 3}
VAL batch total events: 49 | per-stratum: {1: 6, 2: 2, 3: 5, 4: 4, 5: 4, 6: 3, 7: 5, 8: 1, 9: 3, 10: 3, 11: 3, 12: 3, 13: 3, 14: 1, 15: 3}
VAL batch total events: 53 | per-stratum: {1: 3, 2: 6, 3: 3, 4: 5, 5: 1, 6: 4, 7: 4, 8: 2, 9: 5, 10: 2, 11: 3, 12: 1, 13: 6, 14: 2, 15: 6}
VAL batch total events: 46 | per-stratum: {1: 3, 2: 0, 3: 2, 4: 4, 5: 1, 6: 4, 7: 5, 8: 5, 9: 2, 10: 2, 11: 1, 12: 5, 13: 3, 14: 6, 15: 3}
VAL batch total events: 55 | per-stratum: {1: 7, 2: 3, 3: 4, 4: 2, 5: 4, 6: 1, 7: 1, 8: 3, 9: 2, 10: 5, 11: 1, 12: 6, 13: 4, 14: 4, 15: 8}
VAL batch total events: 44 | per-stratum: {1: 5, 2: 0, 3: 4, 4: 1, 5: 6, 6: 1, 7: 1, 8: 4, 9: 5, 10: 3, 11: 3, 12: 4, 13: 1, 14: 3, 15: 3}
VAL batch total events: 42 

In [17]:
import time

# Stratified CoxPH model
model = CoxPHStratified(net, optimizer=optimizer)
model.metrics = {'val_loss': model.loss}
start = time.time() # Record iteration start time
log = model.fit_dataloader(
    train_loader,
    epochs=epochs,
    callbacks=callbacks,
    verbose=True,
    val_dataloader=test_loader  # optional for now
)
stop = time.time() # Record time when training finished
duration = round(stop - start, 2)
print(f"Training time: {duration}")

0:	[0s / 0s],		train_loss: 12.8722,	val_loss: 10.6791
1:	[0s / 0s],		train_loss: 11.7733,	val_loss: 9.7565
2:	[0s / 0s],		train_loss: 10.5729,	val_loss: 9.4788
3:	[0s / 0s],		train_loss: 9.0346,	val_loss: 9.8314
4:	[0s / 0s],		train_loss: 9.4012,	val_loss: 9.1945
5:	[0s / 0s],		train_loss: 9.5483,	val_loss: 9.0401
6:	[0s / 0s],		train_loss: 9.1044,	val_loss: 9.1625
7:	[0s / 0s],		train_loss: 9.7003,	val_loss: 9.1775
8:	[0s / 0s],		train_loss: 8.4436,	val_loss: 8.8804
9:	[0s / 0s],		train_loss: 8.6957,	val_loss: 9.0785
10:	[0s / 0s],		train_loss: 8.9477,	val_loss: 9.2478
11:	[0s / 1s],		train_loss: 8.7793,	val_loss: 8.9633
12:	[0s / 1s],		train_loss: 8.7760,	val_loss: 8.7982
13:	[0s / 1s],		train_loss: 8.5805,	val_loss: 9.1611
14:	[0s / 1s],		train_loss: 9.5282,	val_loss: 8.8223
15:	[0s / 1s],		train_loss: 8.4864,	val_loss: 9.0513
16:	[0s / 1s],		train_loss: 9.1496,	val_loss: 9.1083
17:	[0s / 1s],		train_loss: 8.9383,	val_loss: 8.6480
18:	[0s / 1s],		train_loss: 8.5333,	val_loss: 9.2042

#### *Stratified C-index* 

In [18]:
# ==================== Evaluation ====================
# Convert torch tensors back to numpy objects for evaluation
durations_train_np = durations_train.detach().cpu().numpy()
durations_test_np  = durations_test.detach().cpu().numpy()
events_train_np    = events_train.detach().cpu().numpy()
events_test_np     = events_test.detach().cpu().numpy()
batch_ids_train_np = batch_ids_train.detach().cpu().numpy()
batch_ids_test_np   = batch_ids_test.detach().cpu().numpy()

# Compute baseline hazards (per-batch)
baseline_hazards_strata = model.compute_baseline_hazards(input=x_train, target=(durations_train, events_train), batch_ids=batch_ids_train_np)

# Initialize EvalSurv objects 
tr_surv  = model.predict_surv_df(x_train, batch_ids = batch_ids_train_np)
te_surv = model.predict_surv_df(x_test, batch_ids = batch_ids_test_np)
tr_ev = EvalSurv(tr_surv, durations_train_np, events_train_np, censor_surv='km')
te_ev = EvalSurv(te_surv, durations_test_np, events_test_np, censor_surv='km')

# Concordance index ----------------
tr_strat_c_index  = tr_ev.stratified_concordance_td(batch_indices=batch_ids_train_np) 
te_strat_c_index = te_ev.stratified_concordance_td(batch_indices=batch_ids_test_np) 

print(tr_strat_c_index, te_strat_c_index)

0.84106906778147 0.7787778551532033


In [19]:
# Manual test
from pss.pycox.evaluation.concordance import concordance_td
from pss.pycox.evaluation import ipcw

batch_indices = batch_ids_train.detach().cpu().numpy() if not isinstance(batch_ids_train, np.ndarray) else batch_ids_train
batches = np.unique(batch_indices)
c_index_ls , n_pairs_ls = np.zeros(len(batches)), np.zeros(len(batches))

for i, batch in enumerate(batches):
    # Filter data by batch
    mask = (batch_indices == batch)
    if mask.sum() == 0:
        continue  # skip empty batch
    batch_durations = durations_train_np[mask]
    batch_events = events_train_np[mask]
    batch_surv = tr_ev.surv.iloc[:, mask]
    if batch_events.sum() == 0:
        continue
    
    # Compute concordance for the current batch
    c_index_batch, n_pairs_batch = concordance_td(
        batch_durations, batch_events, batch_surv.values,
        tr_ev.idx_at_times(batch_durations), method='adj_antolini'
    )
    n_pairs_ls[i] = n_pairs_batch
    # n_events_ls[i] = batch_events.sum()
    c_index_ls[i] = c_index_batch
    
print("Final score: %f" % (np.sum(c_index_ls*n_pairs_ls) / np.sum(n_pairs_ls) if np.sum(n_pairs_ls) > 0 else float('nan')))

for e, c in zip(n_pairs_ls, c_index_ls):
    print(f"{int(e)} comparable pairs: {round(c,3)}")

Final score: 0.841069
1848 comparable pairs: 0.847
1954 comparable pairs: 0.864
1909 comparable pairs: 0.831
1909 comparable pairs: 0.763
1977 comparable pairs: 0.826
1752 comparable pairs: 0.876
1915 comparable pairs: 0.82
1833 comparable pairs: 0.879
1960 comparable pairs: 0.774
1998 comparable pairs: 0.882
1921 comparable pairs: 0.868
1661 comparable pairs: 0.869
1710 comparable pairs: 0.847
1848 comparable pairs: 0.857
1792 comparable pairs: 0.82


#### *One-batch C-index* 

In [20]:
baseline_hazards_1batch = model.compute_baseline_hazards(input=x_train, target=(durations_train, events_train))

# Initialize EvalSurv objects 
tr_surv  = model.predict_surv_df(x_train, baseline_hazards_=baseline_hazards_1batch)
te_surv = model.predict_surv_df(x_test, baseline_hazards_=baseline_hazards_1batch)
tr_ev = EvalSurv(tr_surv, durations_train_np, events_train_np, censor_surv='km')
te_ev = EvalSurv(te_surv, durations_test_np, events_test_np, censor_surv='km')

# Concordance index (non-stratified) ----------------
tr_c_index, _  = tr_ev.concordance_td() 
te_c_index, _ = te_ev.concordance_td() 
print(tr_c_index, te_c_index)

0.8173159079858571 0.7819031266405178


#### *Test: Stratified Integrated Brier score*

In [ ]:
# IBS score: stratified ----------------
min_surv = np.ceil(max(np.min(durations_train_np), np.min(durations_test_np)))
max_surv = np.floor(min(np.max(durations_train_np), np.max(durations_test_np)))
times = np.linspace(min_surv, max_surv, 20)
        
tr_strat_ibs = tr_ev.stratified_integrated_brier_score(time_grid=times, batch_indices=batch_ids_train_np) 
te_strat_ibs = te_ev.stratified_integrated_brier_score(time_grid=times, batch_indices=batch_ids_test_np) 

print(tr_strat_ibs, te_strat_ibs)

0.0770192379913959 0.1023440104421342


In [28]:
# Integrated Brier score -----------
tr_brier  = tr_ev.integrated_brier_score(time_grid=times) 
te_brier =  te_ev.integrated_brier_score(time_grid=times)

print(tr_brier, te_brier)

0.08271082420975355 0.09605150927693153


In [ ]:
batch_indices = batch_ids_train.detach().numpy() if not isinstance(batch_ids_train, np.ndarray) else batch_ids_train
batches = np.unique(batch_indices)
brier_ls, n_events_ls = np.zeros(len(batches)), np.zeros(len(batches))

for i, batch in enumerate(batches):
    # Filter data by batch
    mask = (batch_indices == batch)
    if mask.sum() == 0:
        continue  # skip empty batch
    batch_durations = durations_train[mask]
    batch_events = events_train[mask]
    batch_surv = tr_ev.surv.iloc[:, mask]
    if batch_events.sum() == 0:
        continue
    batch_surv_values = tr_ev.surv.values[:, mask]
    batch_censor_surv_values = tr_ev.censor_surv.surv.values[:, mask] 
    # batch_index_surv = tr_ev.index_surv[mask]
    # batch_censor_index_surv = tr_ev.censor_surv.index_surv[mask]
    
    # Compute integrated brier score for the current batch
    brier_batch = ipcw.integrated_brier_score(times, batch_durations, batch_events, 
                                    batch_surv_values, batch_censor_surv_values, 
                                    tr_ev.index_surv, tr_ev.censor_surv.index_surv, np.inf, 
                                    tr_ev.steps, tr_ev.censor_surv.steps)
    n_events_ls[i] = batch_events.sum()
    brier_ls[i] = brier_batch
    
print("Final score: %f\n" % (np.sum(brier_ls*n_events_ls) / np.sum(n_events_ls) if np.sum(n_events_ls) > 0 else float('nan')))

for e, c in zip(n_events_ls, brier_ls):
    print(f"{int(e)} events: {round(c,3)}")

# Pipeline Test

In [ ]:
batchNormType='BE10Asso00_normNone'
dataType='linear-p30'
train_size=5000
random_state=42
time_col='time'
status_col='status'
batch_col='batch_id'
iter_i = 1

train_df, test_df = load_prepared_split(batchNormType=batchNormType,
                                        dataName=dataType,
                                        keep_batch=True,
                                        train_size=train_size,
                                        iter_i=iter_i)

print(f"Training data dimensions: {train_df.shape}")
print(f"Testing data dimensions:  {test_df.shape}")

Training data dimensions: (5000, 541)
Testing data dimensions:  (1000, 541)


In [11]:
# import optuna
# # optuna.get_all_study_names("sqlite:///deepsurv-torch-hp-log.db")]
# optuna.delete_study(storage="sqlite:///deepsurv-torch-hp-log.db",
#                     study_name='BE10Asso00_normNone-linear-moderate-stratified-deepsurv-torch-5000')

In [ ]:
hyperparameters = {
    "num_nodes": {"type": "categorical", "choices": [
        "128", "64", "32", "32-16", "64-32", "128-64", "64-64-32", "32-32-16"
    ]},
    "dropout": {"type": "float", "low": 0.1, "high": 0.5},
    "weight_decay": {"type": "float", "low": 1e-6, "high": 1e-2, "log": True},
    "learning_rate": {"type": "float", "low": 1e-5, "high": 5e-3, "log": True},
    "batch_size": {"type": "categorical", "choices": [256, 128, 64, 32]}
}

ds = DeepSurvPipeline(
    train_df=None, test_df=None,
    batchNormType=batchNormType,
    dataName=dataType,
    time_col=time_col,
    status_col=status_col,
    batch_col=batch_col,
    hyperparameters=hyperparameters,
    is_stratified=False,
    storage_url = "sqlite:///deepsurv-torch-hp-test.db"
)

# optuna.logging.disable_default_handler()
results = train_over_subsets(
    pipeline = ds,
    subset_sizes=[1000],#subset_sizes, 
    runs_per_size=[10],#runs_per_size, 
    splits_per_size=[10],#splits_per_size,
    trials_per_size=[30],#trails_per_size,
    is_tune=True, 
    is_save=False, 
    n_jobs=1,
    trial_threshold=30                              
)
results

Running for training size N=1000...


[I 2026-03-19 09:55:52,094] A new study created in RDB with name: BE10Asso00_normNone-linear-moderate-deepsurv-torch-1000


⚠️No completed trials in Optuna study 'BE10Asso00_normNone-linear-moderate-deepsurv-torch-1000'. Start hyperparameter tuning...


[W 2026-03-19 09:56:14,297] Trial 26 failed with parameters: {'num_nodes': '128', 'dropout': 0.4354219195143624} because of the following error: StorageInternalError('An exception is raised during the commit. This typically happens due to invalid data in the commit, e.g. exceeding max length. ').
Traceback (most recent call last):
  File "/home/nfs/dengy/dl-surv/lib/python3.10/site-packages/sqlalchemy/engine/base.py", line 1967, in _exec_single_context
    self.dialect.do_execute(
  File "/home/nfs/dengy/dl-surv/lib/python3.10/site-packages/sqlalchemy/engine/default.py", line 951, in do_execute
    cursor.execute(statement, parameters)
sqlite3.OperationalError: database is locked

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/nfs/dengy/dl-surv/lib/python3.10/site-packages/optuna/storages/_rdb/storage.py", line 78, in _create_scoped_session
    session.commit()
  File "/home/nfs/dengy/dl-surv/lib/python3.10/site-pa

# ==== Archive ====

In [ ]:
# prepare data
folder = 'linear'
keywords = ['moderate', "latest", 'RW']

train_df, test_df = load_simulate_survival_data(folder=folder, keywords=keywords, test_size=0.2)

train_df.head()

## Feature transforms


In [ ]:
survival_cols = ['time', 'status']

In [ ]:
tr_df, val_df = train_test_split(train_df, 
                                test_size=0.2,
                                shuffle=True, random_state=42,
                                stratify=train_df['status'])

# Transform data
covariate_cols = [col for col in train_df.columns if col not in survival_cols]
standardize = [([col], StandardScaler()) for col in covariate_cols]
leave = [(col, None) for col in survival_cols]
x_mapper = DataFrameMapper(standardize)

# gene expression data
x_train = x_mapper.fit_transform(tr_df[covariate_cols]).astype('float32')
x_val = x_mapper.fit_transform(val_df[covariate_cols]).astype('float32')
x_test = x_mapper.transform(test_df[covariate_cols]).astype('float32')

# prepare labels
get_target = lambda df: (df['time'].values, df['status'].values)
y_train = get_target(tr_df)
y_val = get_target(val_df)
t_test, e_test = get_target(test_df)
val = x_val, y_val

## Neural net

We create a simple MLP with two hidden layers, ReLU activations, batch norm and dropout. 
Here, we just use the `torchtuples.practical.MLPVanilla` net to do this.


In [ ]:
in_features = x_train.shape[1]
num_nodes = [32, 16]
out_features = 1
batch_norm = True
dropout = 0.2
output_bias = True

net = tt.practical.MLPVanilla(in_features, num_nodes, out_features, batch_norm,
                            dropout, output_bias=output_bias)

## Training the model

To train the model we need to define a `torch.optim` optimizer; here we instead use one from `tt.optim` as it has some added functionality.
We use the `Adam` optimizer and set the desired learning rate with `model.lr_finder`.

In [ ]:
optimizer = tt.optim.Adam(weight_decay=0.01)

be_model = CoxPHStratified(net, optimizer)

# we  set it manually to 0.001
be_model.optimizer.set_lr(1e-3)

We include the `EarlyStopping` callback to stop training when the validation loss stops improving. After training, this callback will also load the best performing model in terms of validation loss.

In [ ]:
%%time
batch_size = 64
epochs = 500
callbacks = [tt.callbacks.EarlyStopping(patience=20, min_delta=5e-2)]
verbose = True

batch_indices = np.ones(len(y_train[1]))
log = be_model.fit(x_train, y_train,
                batch_indices,
                batch_size,
                epochs,
                callbacks, 
                verbose=verbose,
                val_data=val, val_batch_size=batch_size
                )